# Distributional boosting (GAMLSS)

This chapter describes how `mboostrs` implements `gamboostLSS`
{cite}`mayr2012generalized,hofner2014gamboostlss` — boosting all parameters
of a response distribution simultaneously. Read it after
[](gamlss_framework.ipynb) (the GAMLSS framework itself) and [](families.ipynb)
(the loss and gradient structure each LSS family implements).

GAMLSS — generalized additive models for location, scale and shape
{cite}`rigby2005generalized` — models **every** parameter of a response
distribution as a function of covariates. Boosting this framework gives
`gamboostLSS`: each distribution parameter $\theta_k$ gets its own additive
predictor $\eta_k$, its own base learners, and its own variable selection.



## The GAMLSS framework

Standard GAMs model the conditional mean $\mu = \mathbb{E}[Y \mid X]$ via an
additive predictor, assuming the rest of the distribution (variance, skewness,
kurtosis) is constant or a simple function of the mean. GAMLSS relaxes this:
**every** parameter of the conditional distribution is modelled as a function of
covariates.

Let the response have conditional density $d(y \mid \theta_1, \dots, \theta_K)$ with
$K$ distributional parameters. For a normal distribution, $K = 2$:
$\theta_1 = \mu$ (location) and $\theta_2 = \sigma$ (scale). For a zero-inflated
negative binomial, $K = 3$: $\theta_1 = \mu$ (mean), $\theta_2 = \sigma$
(dispersion), $\theta_3 = \nu$ (zero-inflation probability). The parameters can
represent **any** property of the distribution: location, scale, shape,
skewness, kurtosis, or mixing probabilities.

Each parameter $\theta_k$ has a monotonic link function $g_k$ and an additive
predictor $\eta_k$:

$$
\theta_k = g_k^{-1}(\eta_k), \qquad
\eta_k(x) = \hat\eta_k^{[0]} + \nu \sum_{m=1}^{M_k} \hat g_k^{[m]}(x).
$$

The link $g_k$ ensures the parameter lies in its natural domain: identity for
real-valued parameters, $\log$ for positive parameters (scale, dispersion),
$\operatorname{logit}$ for probabilities (zero-inflation, Beta precision).

The risk is the joint negative log-likelihood

$$
\mathcal R(\eta_1, \dots, \eta_K) = -\sum_{i=1}^n w_i \,
\log d\big(y_i \mid g_1^{-1}(\eta_1(x_i)), \dots, g_K^{-1}(\eta_K(x_i))\big).
$$

This unified risk drives selection and stopping for **all** parameters
simultaneously.



## The partial gradient

Because the risk depends on all $K$ predictors through the density, the gradient
for parameter $k$ must account for the chain rule through the link:

$$
u_k^{(i)} = -\frac{\partial \mathcal R}{\partial \eta_k^{(i)}}
          = -\frac{\partial \mathcal R}{\partial \theta_k^{(i)}} \,
            \frac{\partial \theta_k^{(i)}}{\partial \eta_k^{(i)}}
          = -\frac{\partial \log d(y_i \mid \boldsymbol\theta)}{\partial \theta_k} \,
            g_k'^{-1}(\eta_k^{(i)}),
$$

where the first factor is the score function for parameter $k$ and the second
factor is the derivative of the inverse link. The other $K-1$ parameters are held
at their current fitted values — the gradient is **partial** to $\eta_k$.

For a parameter with identity link ($g_k(\theta) = \theta$), the chain-rule factor
is 1 and the gradient is directly the score. For a log link ($g_k(\theta) = \log\theta$),
the factor is $\theta = e^{\eta_k}$ because $d\theta/d\eta = e^\eta$.



### Concrete examples

**GaussianLSS** with density $d(y \mid \mu, \sigma) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\big(-\frac{(y-\mu)^2}{2\sigma^2}\big)$:

$$
\begin{aligned}
u_\mu &= -\frac{\partial \mathcal R}{\partial \eta_\mu}
      = \frac{y - \mu}{\sigma^2}, \\[4pt]
u_\sigma &= -\frac{\partial \mathcal R}{\partial \eta_\sigma}
        = -1 + \frac{(y-\mu)^2}{\sigma^2},
\end{aligned}
$$

using $\eta_\sigma = \log\sigma$ so $\partial\sigma/\partial\eta_\sigma = \sigma$.
The $\mu$ gradient is the usual regression residual scaled by the precision
$1/\sigma^2$; the $\sigma$ gradient detects whether the squared residual exceeds
its expected value of $\sigma^2$, driving $\sigma$ up or down accordingly.

**GammaLSS** with log links for both $\mu$ and $\sigma$ (where $\sigma$ is the
Gamma shape parameter):

$$
d(y \mid \mu, \sigma) = \frac{1}{\Gamma(\sigma)} \left(\frac{\sigma}{\mu}\right)^\sigma
                       y^{\sigma-1} e^{-\sigma y/\mu},
$$

$$
\begin{aligned}
u_\mu &= -\sigma\left(\frac{y}{\mu} - 1\right), \\[4pt]
u_\sigma &= -\psi(\sigma) + \log\sigma + 1 - \log\mu + \log y - \frac{y}{\mu},
\end{aligned}
$$

where $\psi(\sigma) = \Gamma'(\sigma)/\Gamma(\sigma)$ is the digamma function.
Note that $u_\mu$ depends on $\sigma$: if the shape changes, the mean gradient
changes scale, which is why the cyclic update order matters.

**ZIPoLSS** (zero-inflated Poisson) has two parameters: $\mu$ (Poisson mean, log
link) and $\sigma$ (zero-inflation probability, logit link). The density is

$$
d(y \mid \mu, \sigma) =
\begin{cases}
\sigma + (1-\sigma) e^{-\mu}, & y = 0, \\[2pt]
(1-\sigma) \frac{\mu^y e^{-\mu}}{y!}, & y > 0.
\end{cases}
$$

The partial gradients involve terms like the probability of being non-zero-inflated
given $y=0$, which is $e^{-\mu}(1-\sigma)/(\sigma + (1-\sigma)e^{-\mu})$.
These are computed in the `ngradient` method of `ZIPoLSS` in `lss.rs`.



## Sequential offset computation

The per-parameter offsets $\hat\eta_k^{[0]}$ are computed **sequentially**:
parameter 0's offset is computed independently, then parameter 1's offset
conditions on parameter 0's offset, and so on. This matches R `gamboostLSS`'s
shared-environment semantics, where the offset routine reads the global
environment for previously-computed parameters.

Formally, let $\boldsymbol\theta^{[0]}$ be the vector of offset-scale parameters.
For $k = 0, \dots, K-1$:

$$
\hat\eta_k^{[0]} = \argmin_{\eta_k} \sum_i w_i \,
\log d\big(y_i \mid \theta_0^{[0]}, \dots, \theta_{k-1}^{[0]},
g_k^{-1}(\eta_k), \theta_{k+1}^{\text{def}}, \dots, \theta_{K-1}^{\text{def}}\big),
$$

where $\theta_{k+1}^{\text{def}}$ are family defaults (e.g., $\sigma = 1$,
$\nu_{\text{df}} = 4$). This sequential conditioning is essential because the
parameters are not orthogonal: the optimal location depends on the scale, and the
optimal scale depends on the location.

**GaussianLSS example:**

1. $\hat\eta_\mu^{[0]} = (\sum w_i y_i) / (\sum w_i)$ — the weighted mean,
   independent of $\sigma$.
2. $\hat\eta_\sigma^{[0]} = \argmin_\sigma \sum_i w_i \big[ \log\sigma +
   (y_i - \hat\mu)^2 / (2\sigma^2) \big]$, which has closed form:

   $$
   \hat\sigma^{[0]} = \sqrt{\frac{\sum w_i (y_i - \hat\mu)^2}{\sum w_i}},
   \quad \hat\eta_\sigma^{[0]} = \log \hat\sigma^{[0]}.
   $$

**StudentTLSS example** (3 parameters: $\mu, \sigma, \nu_{\text{df}}$):

1. $\hat\mu^{[0]}$ is a **robust** location estimate (the weighted median, not
   the mean), computed with $\sigma = 1$ and $\nu_{\text{df}} = 4$ (default df).
   This is because the Student-t mean can be heavily influenced by outliers
   when $\nu_{\text{df}}$ is small.
2. $\hat\sigma^{[0]}$ is estimated conditional on $\hat\mu^{[0]}$,
   again with $\nu_{\text{df}} = 4$.
3. $\hat\nu_{\text{df}}^{[0]}$ is estimated conditional on both
   $\hat\mu^{[0]}$ and $\hat\sigma^{[0]}$.

The `prior` vector in the Rust `LssFamily::offset` method carries the
already-computed link-scale offsets of parameters $0..k$ to the next offset
calculation.

```{warning}
Getting the offset computation order wrong produces visibly different fits. The
code enforces the correct order, and parity tests verify it against R. Do not
reorder the parameter definitions.
```



## Cyclic vs non-cyclic fitting

Two update schedules are implemented, corresponding to two views of how to
optimize the joint risk.



### Cyclic (default)

At each iteration, update **every** parameter once, in order, each using the
others' current fitted values in its partial gradient:
$\mu \to \sigma \to \dots \to \theta_K$, then repeat. Each parameter has its
own $m_{\text{stop},k}$; tuning is over the $K$-dimensional grid.

The cyclic schedule solves, for each full cycle $t = 1, \dots, T$:

$$
\begin{aligned}
\eta_1^{[t]} &= \eta_1^{[t-1]} + \nu \, H_{1, j^\star}
  \big( -\partial\mathcal R / \partial\eta_1 \big|_{\eta_1^{[t-1]}, \dots, \eta_K^{[t-1]}} \big), \\
\eta_2^{[t]} &= \eta_2^{[t-1]} + \nu \, H_{2, j^\star}
  \big( -\partial\mathcal R / \partial\eta_2 \big|_{\eta_1^{[t]}, \eta_2^{[t-1]}, \dots, \eta_K^{[t-1]}} \big), \\
&\vdots \\
\eta_K^{[t]} &= \eta_K^{[t-1]} + \nu \, H_{K, j^\star}
  \big( -\partial\mathcal R / \partial\eta_K \big|_{\eta_1^{[t]}, \dots, \eta_{K-1}^{[t]}, \eta_K^{[t-1]}} \big),
\end{aligned}
$$

where $H_{k, j^\star}$ is the hat matrix of the best-fitting base learner for
parameter $k$. Each parameter's gradient uses the **latest** values of all
parameters — including updates from the current cycle for earlier parameters.
This is a Gauss--Seidel-type strategy: within each cycle, the parameter updates
are sequential, not parallel.

**When to use cyclic:**
- When you want to control each parameter's complexity independently.
- When parameters have different signal-to-noise ratios (e.g. $\mu$ needs
  200 iterations but $\sigma$ needs only 50).
- When you need the full joint tuning grid for interpretability.

Per-parameter `mstop` is set via the `mstop=` argument of `glmboostLSS` /
`gamboostLSS` directly: `mstop={"mu": 200, "sigma": 50}`. Each budget vector
defines its own interleaved path. Once one parameter stops, the remaining
parameters see gradients conditional on its frozen predictor. A model with
unequal budgets therefore generally requires its own fit, rather than prefixes
of a longer equal-budget fit.



### Non-cyclic {cite}`thomas2018gradient`

At each iteration, compute the best update for **every** parameter but apply
only the one that most reduces the **joint** risk. A single $m_{\text{stop}}$
then allocates steps across parameters automatically, collapsing the tuning grid
to one dimension.

Formally, at iteration $m$, for each parameter $k$:

1. Compute the candidate update $\Delta_k = \nu \, H_{k, j^\star} u_k$, where
   $u_k$ is the partial negative gradient at the current joint predictor.
2. Temporarily apply $\Delta_k$ to $\eta_k$ (leaving all other $\eta_\ell$,
   $\ell \ne k$ unchanged).
3. Evaluate the joint risk $\mathcal R_k = \mathcal R(\eta_1, \dots, \eta_k + \Delta_k, \dots, \eta_K)$.
4. The winning parameter is $k^\star = \argmin_k \mathcal R_k$.
5. Apply $\eta_{k^\star} \gets \eta_{k^\star} + \Delta_{k^\star}$ permanently.

The per-parameter selection counts sum to $m_{\text{stop}}$, so one scalar
tunes all parameters. The risk-delta computation is $O(K \cdot n)$ because each
candidate requires evaluating the full density at the perturbed predictor — but
this is still much cheaper than $K$ separate fits.

**When to use non-cyclic:**
- When you want a single tuning parameter (simpler model selection).
- When one parameter dominates the risk reduction early and another later
  (non-cyclic naturally shifts emphasis: it allocates iterations to the
  parameter that most reduces risk at each step).

```{note}
Non-cyclic tends to allocate more iterations to the parameter that is harder to
fit (larger gradient norm) early on, then shift to others once the dominant
parameter's risk is reduced. This can be more efficient than cyclic when the
parameters have gradient scales that differ by orders of magnitude.
```



### Non-cyclic and cyclic fitting with equal mstop

In cyclic fitting with equal $m_{\text{stop}}$, each parameter receives the same
number of updates. In non-cyclic, a parameter that reduces risk more per update
is selected more often. The two schedules therefore produce different fitted
models even when $m_{\text{stop}}$ (for non-cyclic) equals each $m_{\text{stop},k}$
(for cyclic), because the **allocation** across parameters differs.

The non-cyclic schedule has the additional property that it is theoretically
guaranteed to decrease the joint risk at every iteration (or leave it unchanged
if no parameter can improve it), because only a risk-reducing step is taken.
Cyclic fitting, in contrast, may temporarily increase the joint risk within a
cycle because later-parameter updates can undo earlier ones — though this rarely
happens in practice for well-conditioned problems.



## Model selection for distributional models

Cross-validated risk (`cvrisk_lss`) evaluates the joint out-of-bag negative
log-likelihood over a grid of per-parameter $m_{\text{stop}}$ values.
For cyclic models, the grid is $K$-dimensional; for non-cyclic, it is
1-dimensional (a single `mstop`).

Each fold recomputes `df` to `λ` under its training weights. Diagonal grid
points $(m,\ldots,m)$ share checkpoints from one cyclic path. A point with
unequal parameter budgets generally needs a separate fit, because stopping
one parameter changes the subsequent gradients of the others.



### Diagonal checkpoints and unequal budgets

For a path fitted to the requested budget vector $(m_1,\dots,m_K)$, the
predictor for parameter $k$ is reconstructed from its stored increments:

$$
\eta_k^{(m_k)} = \hat\eta_k^{[0]} + \sum_{t=1}^{m_k} \nu_k \, \hat g_{k, j^\star_t},
$$

This reconstruction uses increments from the path fitted with those budgets.
It cannot generally substitute increments from another budget vector.
For diagonal points, all parameters continue through the same complete cycles,
so one fit per fold supplies every requested checkpoint. Non-diagonal points
are refitted. Every grid point also requires an out-of-bag risk evaluation.



### Choosing the grid

For cyclic models, a common strategy is:

1. Fit with a large `mstop` for each parameter (e.g. 200–500).
2. Run `cvrisk` over a grid like `{"mu": range(0, 201, 10), "sigma": range(0, 201, 10)}`.
3. Take the risk-minimizing $(m_\mu, m_\sigma)$ pair.

The grid can be irregular: you might evaluate $\mu$ every 5 steps and
$\sigma$ every 20 if their required budgets differ. Allow for separate fits at
non-diagonal points when choosing the grid size.



### Stability selection for LSS

Stability selection ([](stability_selection.ipynb)) extends to LSS models:
each parameter has its own selection frequency matrix. The `stabsel()` function
accepts LSS fits and returns per-parameter selected sets, each with the same
PFER control. The subsampling and complementary-pairs schemes work unchanged
because the selection is per-parameter (each step selects both a parameter and a
base learner).



## Family catalog

```{list-table}
:header-rows: 1

* - Family
  - Parameters (link)
  - Notes
* - `GaussianLSS`
  - $\mu$ (id), $\sigma$ (log)
  - location–scale normal; $\sigma$ is the residual standard deviation
* - `GammaLSS`
  - $\mu$ (log), $\sigma$ (log)
  - mean–shape; both must be positive
* - `StudentTLSS`
  - $\mu$ (id), $\sigma$ (log), $\mathrm{df}$ (log)
  - heavy tails; $\mathrm{df}$ controls tail weight
* - `BetaLSS`
  - $\mu$ (logit), $\phi$ (log)
  - responses in $(0,1)$; $\phi$ is the precision
* - `NBinomialLSS`
  - $\mu$ (log), $\sigma$ (log)
  - overdispersed counts; NB-II parameterization ($\mathrm{Var} = \mu + \mu^2/\sigma$, $\sigma$ is the size/dispersion)
* - `NegativeBinomialLSS`
  - $\mu$ (log), $\sigma$ (log)
  - a separate family, not an alias of `NBinomialLSS`; NBI parameterization ($\mathrm{Var} = \mu + \sigma\mu^2$); see [risk sign note](#negativebinomiallss-risk-sign-convention)
* - `BCCGLSS`
  - $\mu$ (log), $\sigma$ (log), $\nu$ (id)
  - Box--Cox--Cole--Green; see [nu gradient note](#bccglss-box-cox-cole-green)
* - `BCTLSS`
  - $\mu$ (id), $\sigma$ (log), $\nu$ (id), $\tau$ (log)
  - Box--Cox t for positive responses; $\tau$ controls Student-t tail weight
* - `BCPELSS`
  - $\mu$ (id), $\sigma$ (log), $\nu$ (id), $\tau$ (log)
  - Box--Cox power exponential for positive responses; $\tau$ controls tail shape
* - `JSULSS`
  - $\mu$ (id), $\sigma$ (log), $\nu$ (id), $\tau$ (log)
  - Johnson's SU, centered parameterization matching `gamlss.dist::JSU()`; see [centering note](#jsulss-johnsons-su-distribution)
* - `PIGLSS`
  - $\mu$ (log), $\sigma$ (log)
  - Poisson inverse Gaussian
* - `GumbelLSS`
  - $\mu$ (id), $\sigma$ (log)
  - the right-skewed extreme-value distribution for **maxima**, matching
    `gamlss.dist::RG()`. gamlss's naming is inverted relative to common usage:
    what it calls `GU()` is the extreme-value distribution for *minima* and is
    not implemented here
* - `LogisticLSS`
  - $\mu$ (id), $\sigma$ (log)
  - symmetric, heavier-tailed than Gaussian at the same scale; matches
    `gamlss.dist::LO()`
* - `InverseGaussianLSS`
  - $\mu$ (log), $\sigma$ (log)
  - positive right-skewed responses with $\mathrm{Var} = \sigma^2\mu^3$;
    variance grows with the mean faster than Gamma allows
* - `ZeroAdjustedGammaLSS`
  - $\mu$ (log), $\sigma$ (log), $\nu$ (logit)
  - semicontinuous $y \ge 0$: a Gamma on the positive part, with $\nu$ the
    probability of an exact zero as its own predictor
* - `LogNormalLSS`, `LogLogLSS`, `WeibullLSS`
  - $\mu$ (id), $\sigma$ (log)
  - AFT survival with covariate-varying scale
* - `ZIPoLSS`
  - $\mu$ (log), $\sigma$ (logit)
  - zero-inflated Poisson; $\sigma$ is the zero-inflation probability
* - `ZINBLSS`
  - $\mu$ (log), $\sigma$ (log), $\nu$ (logit)
  - zero-inflated negative binomial; $\nu$ is the zero-inflation probability
* - `DirichletLSS`
  - $\alpha_1,\dots,\alpha_K$ (log)
  - compositional response ($K$ columns summing to 1)
```

```{note}
Several LSS families differ from R `gamboostLSS` in their offsets or gradients:
LogLogLSS's $\sigma$ offset, WeibullLSS's $\mu$ offset, and ZINBLSS's
gradient. The behaviors and their verification are documented in
[](../implementation/parity_and_bugs).
```



## Fisher scoring and the expected information

The partial gradients $u_k$ can be interpreted through the lens of Fisher
scoring. In a classical GLM, the scoring step is

$$
\beta^{(t+1)} = (X^\top W^{(t)} X)^{-1} X^\top W^{(t)} z^{(t)},
$$

where $W$ is the weight matrix (expected information) and $z$ is the working
response. In GAMLSS boosting, each iteration takes a single step in the
direction of the gradient scaled by the inverse Fisher information of the base
learner's hat matrix rather than the full model's.

Specifically, for parameter $k$, the boosting update with linear base learner
$X_k$ is

$$
\Delta\beta_k = \nu \, (X_k^\top W X_k + \lambda_k K_k)^{-1} X_k^\top W u_k,
$$

where $u_k$ is the score vector (gradient of the log-likelihood). If $W$ were
the expected information matrix $\mathbb{E}[-\partial^2 \ell / \partial\eta_k^2]$
and $\lambda_k = 0$, this would be exactly one Fisher scoring step for
parameter $k$, treating the other parameters as known.

The GAMLSS boosting engine therefore implements a **penalized, step-length
damped** version of Fisher scoring, where the base learners provide the design
and the penalty, the family provides the score $u_k$, and the engine limits the
step via $\nu$ and the penalty shrinkage $\lambda_k$.



## Zero-inflated families: theory and gradients

Zero-inflated families (`ZIPoLSS`, `ZINBLSS`) model count data with excess
zeros beyond what the base distribution predicts. They are two-component
mixtures: a degenerate distribution at zero (with probability $\pi$) and a
count distribution (Poisson or negative binomial, with probability $1 - \pi$).

For `ZIPoLSS`, the density for observation $i$ is

$$
P(Y = y \mid \mu_i, \pi_i) =
\begin{cases}
\pi_i + (1 - \pi_i) e^{-\mu_i}, & y = 0, \\[4pt]
(1 - \pi_i) \frac{\mu_i^{y} e^{-\mu_i}}{y!}, & y > 0,
\end{cases}
$$

where $\mu_i = e^{\eta_{\mu,i}}$ (log link) and
$\pi_i = \operatorname{logit}^{-1}(\eta_{\sigma,i}) = 1/(1+e^{-\eta_{\sigma,i}})$
(logit link). The negative log-likelihood for a zero observation
is $-\log(\pi + (1-\pi)e^{-\mu})$, which involves both parameters — the
zero-inflation probability $\pi$ and the Poisson mean $\mu$ compete to explain
zeros. The partial gradients are:

$$
\begin{aligned}
u_\mu^{(i)} &= -\frac{\partial \ell_i}{\partial\eta_\mu}
            = \mu_i \left[ 1 - \frac{y_i}{\mu_i} - \frac{1 - \pi_i}{\pi_i + (1-\pi_i)e^{-\mu_i}} \mathbf{1}\{y_i=0\} \right], \\[4pt]
u_\sigma^{(i)} &= -\frac{\partial \ell_i}{\partial\eta_\sigma}
              = \pi_i(1-\pi_i) \left[ \frac{1 - e^{-\mu_i}}{\pi_i + (1-\pi_i)e^{-\mu_i}} \mathbf{1}\{y_i=0\} - \mathbf{1}\{y_i>0\} \right].
\end{aligned}
$$

The term $\frac{1-\pi_i}{\pi_i + (1-\pi_i)e^{-\mu_i}}$ is the posterior
probability that a zero observation comes from the Poisson component rather
than the zero-inflation component — the "responsibility" of the count model
for each zero. This EM-like structure carries over directly to gradient
boosting: the gradient assigns each zero observation's responsibility
between the two sources.

`ZINBLSS` adds a dispersion parameter $\sigma$ (log link) to the negative
binomial component, giving three parameters $(\mu, \sigma, \nu)$ where $\nu$
is the zero-inflation probability (logit link). The NB2 density for the count
component is

$$
P_{\text{NB}}(Y = y \mid \mu, \sigma) = \binom{y + \sigma - 1}{y}
\left(\frac{\sigma}{\mu + \sigma}\right)^\sigma
\left(\frac{\mu}{\mu + \sigma}\right)^y,
$$

and the zero-inflated density is $P(y) = \nu \mathbf{1}\{y=0\} + (1-\nu) P_{\text{NB}}(y)$.
The gradient for $\mu$ involves the NB score plus a correction for zero
observations, and the gradient for $\sigma$ involves the digamma function.
The R implementation in `gamboostLSS` computes the ZINB gradient with a
scalar-recycled NB density at $y=0$ instead of a per-observation value;
`mboostrs` computes the per-observation gradient.



## Dirichlet regression for compositional data

`DirichletLSS` models a $K$-component compositional response
$\mathbf{y} = (y_1, \dots, y_K)$ where $y_j > 0$ and $\sum_j y_j = 1$.
Each component $j$ has its own additive predictor $\eta_j$ with log link,
parameterizing the Dirichlet concentration $\alpha_j = e^{\eta_j}$:

$$
d(\mathbf{y} \mid \boldsymbol\alpha) = \frac{1}{B(\boldsymbol\alpha)}
\prod_{j=1}^K y_j^{\alpha_j - 1}, \qquad
B(\boldsymbol\alpha) = \frac{\prod_{j=1}^K \Gamma(\alpha_j)}{\Gamma(\sum_j \alpha_j)}.
$$

The negative log-likelihood per observation is

$$
\ell_i = \log B(\boldsymbol\alpha_i) - \sum_{j=1}^K (\alpha_{ij} - 1) \log y_{ij},
$$

and the partial gradient for component $j$ is

$$
u_j^{(i)} = \alpha_{ij} \big[ \psi(\alpha_{ij}) - \psi(\textstyle\sum_\ell \alpha_{i\ell}) - \log y_{ij} \big],
$$

where $\psi$ is the digamma function. The `DirichletLSS` implementation in
`lss.rs` stores the response as a concatenated $K$-column slice
$[y_{11}, \dots, y_{1K}, y_{21}, \dots, y_{nK}]$, deriving the observation
count $n$ from the weight vector length divided by $K$. Each of the $K$
parameters gets its own set of base learners, but the joint risk couples
them through the $\log B(\boldsymbol\alpha)$ term.



## Interpreting `LSS` fits

Each distribution parameter has its own selection path, coefficient vector, and
partial effects — analogous to a single-predictor fit:

```python
m = gamboostLSS("y ~ bbs(x, df=4)", data=data, families=GaussianLSS(), ...)
m.predict(parameter="mu", type="response")    # location at each x
m.predict(parameter="sigma", type="response")  # scale at each x
m.coef(parameter="mu")    # coefficients of the mu model
m.coef(parameter="sigma") # coefficients of the sigma model
```


The `predict(type="response")` applies the inverse link to each parameter's
linear predictor, returning the natural-scale prediction. For `GaussianLSS`,
this gives the predicted mean and predicted standard deviation at each
observation — a full conditional distribution.

The parameters are not independent: a change in $\mu$ affects the optimal
$\sigma$ (especially for families where the variance depends on the mean,
like Gamma or NBinomial), and vice versa. The cyclic or non-cyclic schedule
accounts for this coupling by re-evaluating each parameter's gradient in the
context of the others' current values.



## Parameterization notes

Several `mboostrs` LSS families use parameterizations that differ from the
most common textbook form, matching R `gamlss.dist` instead. These notes
document the differences and their practical consequences for anyone
comparing `mboostrs` output to results from other software.


(bccglss-box-cox-cole-green)=

### BCCGLSS: Box--Cox--Cole--Green

`BCCGLSS` has three parameters: $\mu$ (log link), $\sigma$ (log link), and
$\nu$ (identity link). The standardized residual is

$$
z = \frac{(y/\mu)^\nu - 1}{\nu\sigma}, \qquad (\text{with the log limit when } \nu \to 0).
$$

BCCG is supported on $y > 0$, so the Box-Cox transform maps it to a normal
truncated below and the density carries a normalizer $1/\Phi(1/(\sigma|\nu|))$.
That factor depends on $\sigma$ and $\nu$, so it belongs in their gradients as
well as in the risk:

$$
u_\nu = (z - \ell/\sigma) \cdot (z/\nu) - \ell \cdot (z^2 - 1)
      + \operatorname{sign}(\nu)\,\frac{h}{\sigma\nu^2},
\qquad
h = \frac{\phi(1/(\sigma|\nu|))}{\Phi(1/(\sigma|\nu|))},
$$

with $\ell = \log(y/\mu)$, and correspondingly $h/(\sigma|\nu|)$ added to the
$\sigma$ gradient on the predictor scale.

```{note}
The truncation term is included in the risk and both gradients. It is small
when $\sigma|\nu|$ is small but becomes material for larger values: over five
points at $\sigma = 1.5$, $\nu = 1$, omitting it changes the risk by 1.46.
With the term included, `risk()` matches `gamlss.dist::dBCCG` to $10^{-13}$
across $\sigma \in [0.1, 1.5]$.

These gradients agree with `gamlss.dist::BCCG`'s. One difference remains:
gamlss's `dldv` returns `NaN` at $\nu = 0$, where the limit exists and is
finite,

$$
\lim_{\nu \to 0} \frac{\partial \ell}{\partial \nu}
  = \ell_y - \frac{\ell_y^3}{2\sigma^2},
\qquad \ell_y = \log(y/\mu),
$$

which `mboostrs` returns. Because the $\nu$ offset is zero, the fit needs
the limiting score at initialization to update $\nu$.
```


(bct-bcpe-box-cox-tail-families)=

### BCTLSS and BCPELSS: Box--Cox models with fitted tails

Both families model a strictly positive response through the standardized
Box--Cox residual

$$
z = \frac{(y/\mu)^\nu - 1}{\nu\sigma}, \qquad
z = \frac{\log(y/\mu)}{\sigma} \text{ when } \nu=0.
$$

Here $\mu>0$ locates the distribution, $\sigma>0$ sets relative spread,
and $\nu$ changes the Box--Cox transformation and hence skewness. Because
$y$ is restricted to be positive, the distribution of $z$ is truncated at
$-1/(\sigma\nu)$ when $\nu>0$ or at $1/(\sigma|\nu|)$ when $\nu<0$.
The likelihood includes the corresponding truncation normalizer. In
particular, $\mu$ is not generally the response mean, so `mean()` refuses
these families rather than treating the location curve as a mean curve.

`BCTLSS` assigns $z$ a truncated Student-t distribution with $\tau$ degrees
of freedom. Smaller $\tau$ gives heavier tails; as $\tau$ grows, the kernel
approaches a normal density. `BCPELSS` instead uses a variance-standardized
power-exponential kernel proportional to
$\exp[-|z/c(\tau)|^{\tau}/2]$. At $\tau=2$ this kernel is normal; values
below 2 have heavier tails and values above 2 have lighter tails. Thus the
two families answer different modeling questions: BCT uses a familiar
Student-t tail parameter, while BCPE permits both heavier- and lighter-than-
normal tails.

All four parameter predictors can contain base learners. Use fitted centile
curves or PIT diagnostics to assess whether covariate-dependent scale,
skewness, and tail shape are needed; a lower training risk alone does not
show that the extra parameter functions generalize. `BCTLSS` reproduces the
forward-difference $\tau$ score used by `gamlss.dist::BCT`, so boosting paths
match `gamboostLSS` as well as the density and quantile definitions.


(jsulss-johnsons-su-distribution)=

### JSULSS: Johnson's SU distribution

`JSULSS` has four parameters ($\mu$, $\sigma$, $\nu$, $\tau$) and uses the
**centered** parameterization from `gamlss.dist::JSU()`, where $\mu$ *is*
the mean and $\sigma$ *is* the standard deviation of the distribution. The
centering involves constants:

$$
\omega = -\nu/\tau, \quad
w = e^{1/\tau^2}, \quad
c = \frac{1}{\sqrt{\tfrac12(w-1)(w\cosh 2\omega + 1)}}, \quad
\mu_c = \mu + c\sigma\sqrt{w}\sinh\omega, \quad
\sigma_c = c\sigma,
$$

with $z = (y - \mu_c)/\sigma_c$ and $r = -\nu + \operatorname{asinh}(z) \cdot \tau$.
The density is

$$
f(y) = \frac{\tau}{c\sigma\sqrt{2\pi}} \cdot \frac{1}{\sqrt{1 + z^2}}
      \cdot e^{-r^2/2}.
$$

Without the centering constants $c$, $\mu_c$, $\sigma_c$ (the uncentered
parameterization), $\mu$ and $\sigma$ are not the mean and standard
deviation — they are the location and scale of the *generating* normal
distribution. Coefficients from the two parameterizations differ
substantially, so comparing with R requires using the same parameterization.
`mboostrs` uses the centered form to match R.


(negativebinomiallss-risk-sign-convention)=

### NegativeBinomialLSS: risk sign convention

In `mboostrs`, the `risk()` function returns the **negative**
log-likelihood (NLL), not the log-likelihood. This is required because the
ngradient is defined as $-\partial(\mathrm{NLL})/\partial\eta$: if `risk()`
returned the positive log-likelihood, the finite-difference gradient check
would fail (the sign would be reversed). This convention matters when comparing risk values across packages. All `mboostrs` LSS families follow this
NLL-sign convention consistently.



### B-spline basis column count for Bconstrained

When constructing a constraint matrix $C$ for `bconstrained`, the B-spline
basis column count is $n_{\text{inner}} + \text{degree} + 1$, where
$n_{\text{inner}}$ is the `knots` parameter (default 20). With default
settings (`knots=20`, `degree=3`), this gives $20 + 3 + 1 = 24$ columns,
so $C$ must have 24 columns. If `center=TRUE` is used, the column count
drops by `differences` (default 2), giving 22.



## Exercises

````{admonition} Exercise: Derive the GaussianLSS $\sigma$ partial gradient
:class: dropdown

The GaussianLSS density is

$$
d(y \mid \mu, \sigma) = \frac{1}{\sqrt{2\pi\sigma^2}}
\exp\!\left(-\frac{(y-\mu)^2}{2\sigma^2}\right),
$$

with link functions $\eta_\mu = \mu$ (identity) and $\eta_\sigma = \log\sigma$
(log).  The negative log-likelihood (risk) for a single observation is

$$
\ell(y, \mu, \sigma) = \frac12 \log(2\pi) + \log\sigma
                      + \frac{(y-\mu)^2}{2\sigma^2}.
$$

**a)** Compute $\partial\ell/\partial\sigma$ and $\partial\sigma/\partial\eta_\sigma$.
Then use the chain rule to derive $u_\sigma = -\partial\ell/\partial\eta_\sigma$.

**b)** Show that $u_\sigma = -1 + (y-\mu)^2 / \sigma^2$.

**c)** Interpret the gradient: when is $u_\sigma$ positive (driving $\sigma$ up)
and when is it negative (driving $\sigma$ down)?  What does this imply about
how the boosting algorithm adjusts the scale?

**d)** Repeat the derivation for $u_\mu$, confirming the result
$u_\mu = (y-\mu)/\sigma^2$ given in the chapter.

````

````{admonition} Solution
:class: dropdown

**a)** The partial derivatives:

$$
\frac{\partial\ell}{\partial\sigma}
= \frac{1}{\sigma} - \frac{(y-\mu)^2}{\sigma^3},
\qquad
\frac{\partial\sigma}{\partial\eta_\sigma}
= \frac{\partial}{\partial\eta_\sigma} e^{\eta_\sigma}
= e^{\eta_\sigma} = \sigma.
$$

By the chain rule,

$$
u_\sigma = -\frac{\partial\ell}{\partial\eta_\sigma}
= -\frac{\partial\ell}{\partial\sigma} \cdot \frac{\partial\sigma}{\partial\eta_\sigma}
= -\Big(\frac{1}{\sigma} - \frac{(y-\mu)^2}{\sigma^3}\Big)\,\sigma
= -1 + \frac{(y-\mu)^2}{\sigma^2}.
$$

**b)** This is the expression from part (a).

**c)** $u_\sigma > 0$ when $(y-\mu)^2 > \sigma^2$ — the squared residual exceeds
the current scale estimate — so the algorithm increases $\sigma$ to accommodate
the larger scatter.  $u_\sigma < 0$ when $(y-\mu)^2 < \sigma^2$, signalling the
scale is too large, and the algorithm shrinks $\sigma$.  At the conditional MLE
(when the squared residual equals $\sigma^2$ on average), the gradient is zero.
This is exactly the score function for the scale parameter of a normal
distribution.

**d)** For $\mu$, the chain rule is trivial ($\partial\mu/\partial\eta_\mu = 1$):

$$
\frac{\partial\ell}{\partial\mu}
= -\frac{y-\mu}{\sigma^2},
\qquad
u_\mu = -\frac{\partial\ell}{\partial\eta_\mu}
= \frac{y-\mu}{\sigma^2}.
$$

The gradient is the raw residual scaled by the precision $1/\sigma^2$:
observations with small $\sigma$ (high precision) exert more influence on the
location update.
````

````{admonition} Exercise: Cyclic vs non-cyclic equivalence for orthogonal parameters
:class: dropdown

Consider a two-parameter GAMLSS model ($\theta_1, \theta_2$) whose parameters
are **Fisher-orthogonal**: the cross-derivative of the log-likelihood is zero
everywhere:

$$
\mathbb{E}\!\left[ \frac{\partial^2\ell}{\partial\eta_1\partial\eta_2} \right] = 0
\quad\Longleftrightarrow\quad
\frac{\partial^2\ell}{\partial\eta_1\partial\eta_2} = 0\ \text{(observed)}.
$$

**a)** Show that under orthogonality, the cyclic update for $\eta_1$ at iteration
$t$ using $\eta_2^{[t-1]}$ is identical to using $\eta_2^{[t]}$ (i.e. the order
within a cycle does not matter).

**b)** Show that the non-cyclic risk comparison reduces to comparing squared
gradient norms: the selected parameter $k^\star$ satisfies

$$
k^\star = \argmin_k \mathcal{R}(\eta_1, \dots, \eta_k + \Delta_k, \dots, \eta_K)
        = \argmax_k \|\Delta_k\|^2 / (\text{step cost}),
$$

and when both learners have the same effective df, this is simply
$\argmax_k \|u_k\|^2$.

**c)** Conclude that when the parameters are orthogonal and all base learners
have identical $df$, the cyclic and non-cyclic schedules produce the same
sequence of updates.  Why does correlation between parameters break this
equivalence?

````

````{admonition} Solution
:class: dropdown

**a)** Under orthogonality, $\partial u_1/\partial\eta_2 = 0$: the gradient for
$\eta_1$ does not depend on $\eta_2$.  Hence evaluating $u_1$ at
$(\eta_1^{[t-1]}, \eta_2^{[t-1]})$ or at $(\eta_1^{[t-1]}, \eta_2^{[t]})$
gives the same result.  In cyclic fitting the order is irrelevant.

**b)** For a small step $\Delta_k$, a second-order Taylor expansion gives:

$$
\mathcal{R}(\eta_k + \Delta_k) - \mathcal{R}(\eta_k)
\approx -\langle u_k, \Delta_k\rangle + \tfrac12 \Delta_k^\top \mathcal{I}_k \Delta_k,
$$

where $\mathcal{I}_k$ is the Fisher information for $\eta_k$.  When the base
learner is a linear smoother with hat matrix $H_k$, the optimal update is
$\Delta_k \propto H_k u_k$, and the risk reduction is proportional to
$u_k^\top H_k u_k$.  If all learners have the same df (hence similar spectral
properties), the reduction is dominated by $\|u_k\|^2$, so the parameter with
the largest gradient norm is selected.

**c)** Under orthogonality, each parameter's gradient depends only on its own
current value, not on the other parameters' latest values.  Cyclic fitting
(with any order) and non-cyclic selection (which picks the parameter with
largest gradient norm) therefore make identical decisions at every step:
both update the parameter with the largest gradient, and the update magnitude
is the same.

When parameters are correlated, the gradient for $\eta_1$ changes after
$\eta_2$ is updated.  Cyclic fitting captures this within-cycle interaction
(updating $\eta_2$ before evaluating $u_1$), while non-cyclic evaluates all
gradients at the *same* joint predictor.  The two schedules then diverge.
````

````{admonition} Exercise: Prove joint-risk monotonicity in non-cyclic GAMLSS
:class: dropdown

The non-cyclic GAMLSS schedule at iteration $t$:

1. Compute candidate updates $\Delta_k = \nu \, H_{k, j^\star} \, u_k$ for
   every parameter $k = 1, \dots, K$, all evaluated at the current joint
   predictor $(\eta_1^{[t-1]}, \dots, \eta_K^{[t-1]})$.
2. For each $k$, compute the joint risk after applying only $\Delta_k$:
   $\mathcal{R}_k = \mathcal{R}(\eta_1^{[t-1]}, \dots, \eta_k^{[t-1]} + \Delta_k, \dots, \eta_K^{[t-1]})$.
3. Select $k^\star = \argmin_k \mathcal{R}_k$ and apply $\eta_{k^\star} \gets \eta_{k^\star} + \Delta_{k^\star}$.

**a)** Show that if $\nu$ is small enough, every candidate update decreases (or
does not increase) the joint risk: $\mathcal{R}_k \le \mathcal{R}^{[t-1]}$ for
all $k$.

**b)** Hence prove that the non-cyclic schedule guarantees
$\mathcal{R}^{[t]} \le \mathcal{R}^{[t-1]}$ — monotone joint-risk decrease.

**c)** Construct a simple 2-parameter cyclic example where $\mathcal{R}$
could *increase* within a cycle.  (Hint: consider parameters that compete to
explain the same variation, like $\mu$ and a zero-inflation probability for
zeros.)

**d)** Explain why this monotonicity makes non-cyclic more predictable and
allows a single $m_{\text{stop}}$ to control all parameters, while cyclic
fitting needs a $K$-dimensional grid.

````

````{admonition} Solution
:class: dropdown

**a)** The candidate $\Delta_k$ is a (scaled) descent direction: it points
along $H_k u_k$, which is the projection of the negative gradient onto the
base learner's column space.  For sufficiently small $\nu$, the linear term
dominates the higher-order terms in the Taylor expansion:

$$
\mathcal{R}_k - \mathcal{R}^{[t-1]}
= -\nu\, u_k^\top H_k u_k + O(\nu^2).
$$

Since $H_k$ is positive semi-definite (it is a hat matrix),
$u_k^\top H_k u_k \ge 0$.  Hence for $\nu$ small enough,
$\mathcal{R}_k - \mathcal{R}^{[t-1]} \le 0$.

**b)** The selected $k^\star$ is the parameter giving the smallest
$\mathcal{R}_k$.  From (a), every $\mathcal{R}_k \le \mathcal{R}^{[t-1]}$,
so $\mathcal{R}_{k^\star} \le \mathcal{R}^{[t-1]}$.  Hence
$\mathcal{R}^{[t]} \le \mathcal{R}^{[t-1]}$ — the joint risk never increases.
(In practice, $\nu = 0.1$ is easily small enough for this to hold.)

**c)** Consider ZIPoLSS with $\mu$ (Poisson mean) and $\sigma$ (zero-inflation
probability).  If the data have many zeros, updating $\mu$ first may increase
$\mu$ to fit the zeros, which *increases* the risk.  Then updating $\sigma$
increases the zero-inflation probability to compensate, *decreasing* the risk
below the original level.  The cycle-level net effect is a decrease, but the
within-cycle $\mu$-only step temporarily increased the risk.  Non-cyclic
would never take the $\mu$-only step alone because it would increase the risk.

**d)** Monotonicity means the joint risk is a well-behaved function of the
iteration count: it always decreases (or stays flat), never oscillates.  This
allows a single `mstop` to index the entire path — you can slice the path at
any $m$ and get a model whose risk is no worse than any earlier model.  In
cyclic fitting, the risk can fluctuate within cycles, so you must record
*which* parameter was updated when, requiring separate $m_{\text{stop},k}$
values to index the useful models.
````


## Verifying the engine: per-parameter gradients

The same idea from [](functional_gradient_boosting.ipynb) — finite-difference
the risk and compare to the analytic gradient — extends to LSS families via
`mboostrs._rust.lss_family_diagnostics(name, n_params, y, etas, w)`, which
calls `LssFamily::offset`/`ngradient`/`risk` directly (not the fit/predict
hot path). Because each parameter's gradient is *partial* (holding the other
$K-1$ predictors fixed), the check perturbs one parameter's `eta` at a time
and compares against $-\partial\mathcal R/\partial\eta_k$ with the others held
fixed — exactly the partial-derivative claim in the algorithm box above.


In [1]:
import numpy as np
import polars as pl
import mboostrs as mb
import mboostrs._rust as _rust

rng = np.random.default_rng(0)
n = 8
w = np.ones(n)


def lss_gradient_ratio(name, n_params, y, etas, w, h=1e-6):
    """Per-parameter grad[k] / (-d(risk)/d(eta_k)), holding the other
    parameters' etas fixed; entries with a near-zero derivative are dropped."""
    _, grads, _ = _rust.lss_family_diagnostics(name, n_params, y, etas, w)
    ratios = []
    for k in range(n_params):
        raw = np.zeros_like(etas[k])
        for i in range(len(etas[k])):
            etas_p, etas_m = [e.copy() for e in etas], [e.copy() for e in etas]
            etas_p[k][i] += h
            etas_m[k][i] -= h
            _, _, r_plus = _rust.lss_family_diagnostics(name, n_params, y, etas_p, w)
            _, _, r_minus = _rust.lss_family_diagnostics(name, n_params, y, etas_m, w)
            raw[i] = -(r_plus - r_minus) / (2 * h)
        mask = np.abs(raw) > 1e-8
        ratios.append(grads[k][mask] / raw[mask])
    return ratios


# (family, n_params, response generator) -- one representative y per family,
# respecting each family's response domain.
y_real = rng.normal(size=n)
y_pos = rng.uniform(0.5, 5, size=n)
y_count = rng.integers(1, 10, size=n).astype(float)
y_count0 = rng.integers(0, 10, size=n).astype(float)
y_prop = rng.uniform(0.1, 0.9, size=n)
time, status = rng.uniform(0.5, 5, size=n), rng.choice([0.0, 1.0], size=n)
y_surv = np.concatenate([time, status])
y_dir = rng.dirichlet(np.ones(3), size=n).ravel()
# semicontinuous: exact zeros mixed with a positive right-skewed part
y_zero_adj = np.where(rng.random(n) < 0.4, 0.0, rng.uniform(0.5, 5, size=n))

lss_cases = {
    "GaussianLSS": (2, y_real), "GammaLSS": (2, y_pos), "StudentTLSS": (3, y_real),
    "BetaLSS": (2, y_prop), "NBinomialLSS": (2, y_count),
    "NegativeBinomialLSS": (2, y_count), "PoissonLSS": (1, y_count),
    "GeometricLSS": (1, y_count0), "BinomialLSS": (1, y_prop),
    "GumbelLSS": (2, y_real), "Pareto2LSS": (2, y_pos), "BCCGLSS": (3, y_pos),
    "BCTLSS": (4, y_pos), "BCPELSS": (4, y_pos), "JSULSS": (4, y_real),
    "ZIPoLSS": (2, y_count0), "ZINBLSS": (3, y_count0),
    "ZeroAdjustedPoissonLSS": (2, y_count0), "LogisticLSS": (2, y_real),
    "InverseGaussianLSS": (2, y_pos), "ZeroAdjustedGammaLSS": (3, y_zero_adj),
    "LogNormalLSS": (2, y_surv),
    "LogLogLSS": (2, y_surv), "WeibullLSS": (2, y_surv), "DirichletLSS": (3, y_dir),
}

rows = []
for cname, (k, y) in lss_cases.items():
    name = cname.lower()
    etas = [rng.normal(size=len(y) if y is not y_surv else n) * 0.2 for _ in range(k)]
    if cname == "DirichletLSS":
        etas = [rng.normal(size=n) * 0.2 for _ in range(k)]
    ratios = lss_gradient_ratio(name, k, y, etas, w)
    flat = np.concatenate(ratios)
    rows.append({"family": cname, "n_params": k, "ratio (mean)": flat.mean(),
                 "ratio (std)": flat.std(), "consistent": bool(flat.std() < 1e-4)})

pl.DataFrame(rows)


/tmp/ipykernel_67697/3875936267.py:65: RuntimeWarning: Mean of empty slice
  rows.append({"family": cname, "n_params": k, "ratio (mean)": flat.mean(),
/Users/fcela/src/public_github/mboostrs/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/Users/fcela/src/public_github/mboostrs/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:219: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/fcela/src/public_github/mboostrs/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:178: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/Users/fcela/src/public_github/mboostrs/.venv/lib/python3.12/site-packages/numpy/_core/_methods.py:211: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


family,n_params,ratio (mean),ratio (std),consistent
str,i64,f64,f64,bool
"""GaussianLSS""",2,1.0,1.2044e-9,true
"""GammaLSS""",2,1.0,2.5990e-9,true
"""StudentTLSS""",3,1.0,4.1137e-9,true
"""BetaLSS""",2,1.0,1.3923e-9,true
"""NBinomialLSS""",2,1.0,5.0742e-8,true
…,…,…,…,…
"""ZeroAdjustedGammaLSS""",3,1.0,2.7269e-9,true
"""LogNormalLSS""",2,1.0,1.3873e-9,true
"""LogLogLSS""",2,1.0,5.9879e-10,true


Twenty-four of the 25 families in the table have ratio $=1$ up to numerical
differentiation error. `BCTLSS` is the documented exception: its $\tau$
score reproduces the finite forward-difference expression in
`gamlss.dist::BCT`, which is not exactly the derivative of the reported
log-likelihood. Its other three scores, and every `BCPELSS` score, pass the
central finite-difference check. A separate parity test pins the BCT $\tau$
score to R so that a mathematically cleaner derivative cannot silently change
the fitted boosting path. `PIGLSS` (Poisson inverse Gaussian)
is not in the table above: it raises `NotImplementedError` (requires the
Bessel K function).

### A grouped effect on scale

A random-intercept learner can belong to any distribution parameter, not only
the location predictor. In the next model, `mu` varies smoothly with `x`, while
`sigma` contains `Brandom(group)`. The latter is one candidate learner whose
coefficient block contains one ridge-penalized value per group. Selecting that
learner therefore changes the fitted conditional spread for a group without
requiring four unrelated formula terms.


In [2]:
rng = np.random.default_rng(17)
n = 480
group = np.arange(n) % 4
x = rng.uniform(-2, 2, n)
z = rng.normal(size=n)
sigma_group = np.array([0.45, 0.70, 1.05, 1.50])
mu = 0.8 * np.sin(x) + 0.25 * z
y = rng.normal(mu, sigma_group[group])
grouped = pl.DataFrame({"y": y, "x": x, "z": z, "group": group})

grouped_model = mb.gamboostLSS(
    {
        "mu": "y ~ bbs(x, df=4) + bols(z)",
        "sigma": "y ~ brandom(group, df=3) + bols(z)",
    },
    data=grouped, families=mb.GaussianLSS(),
    control=mb.boost_control(mstop=120, nu=0.1),
)
probe = pl.DataFrame({"group": np.arange(4), "z": np.zeros(4)})
pl.DataFrame({
    "group": np.arange(4),
    "true_sigma": sigma_group,
    "fitted_sigma_at_z0": grouped_model.predict(
        parameter="sigma", type="response", newdata=probe
    ),
})


group,true_sigma,fitted_sigma_at_z0
i64,f64,f64
0,0.45,0.476589
1,0.7,0.747992
2,1.05,1.116089
3,1.5,1.537117


Because `GaussianLSS` uses a log link for scale, each fitted group
coefficient is additive on $\eta_\sigma$ and multiplicative on $\sigma$.
The table compares the resulting standard deviations: groups with the same
location curve can have different conditional spreads. Prediction accepts the integer codes observed during
fitting. A new code raises an explicit unknown-level error; it is not assigned a
zero effect. CV and stability selection rebuild the weighted centering and ridge
factorization for each resample while reusing the indicator layout.

### Cyclic vs non-cyclic in practice

Non-cyclic fitting (`method="noncyclic"`) is available for both
`glmboostLSS` and mixed-learner `gamboostLSS` models. The example below uses
linear learners so that the schedule, rather than the learner basis, is the
only changing part of the comparison. Both fits receive the same total step
budget.


In [3]:
import polars as pl

rng = np.random.default_rng(1)
n = 300
x1, x2, x3 = rng.normal(size=n), rng.normal(size=n), rng.normal(size=n)
mu_true = 1 + 2 * x1 - 0.5 * x2
sigma_true = np.exp(0.3 + 0.4 * x3)
y_sim = rng.normal(mu_true, sigma_true)
sim = pl.DataFrame({"y": y_sim, "x1": x1, "x2": x2, "x3": x3})

ctrl = mb.boost_control(mstop=200, nu=0.1)
m_cyclic = mb.glmboostLSS("y ~ x1 + x2 + x3", data=sim, families=mb.GaussianLSS(),
                          control=ctrl, method="cyclic")
m_noncyclic = mb.glmboostLSS("y ~ x1 + x2 + x3", data=sim, families=mb.GaussianLSS(),
                             control=ctrl, method="noncyclic")

sel_mu = m_noncyclic.selected(parameter="mu")
sel_sigma = m_noncyclic.selected(parameter="sigma")
print(f"noncyclic risk monotone decreasing: {np.all(np.diff(m_noncyclic.risk_) <= 1e-9)}")
print(f"noncyclic step allocation: mu={len(sel_mu)}, sigma={len(sel_sigma)} (of {ctrl[0]} total)")
print(f"cyclic step allocation:    mu={ctrl[0]}, sigma={ctrl[0]} (each parameter gets the full budget)")
print(f"\nfinal risk -- cyclic: {m_cyclic.risk_[-1]:.3f}, noncyclic: {m_noncyclic.risk_[-1]:.3f}")
print(f"mu coefficients   -- cyclic: {m_cyclic.coef(parameter='mu')}")
print(f"mu coefficients   -- noncyclic: {m_noncyclic.coef(parameter='mu')}")


noncyclic risk monotone decreasing: True
noncyclic step allocation: mu=134, sigma=66 (of 200 total)
cyclic step allocation:    mu=200, sigma=200 (each parameter gets the full budget)

final risk -- cyclic: 507.142, noncyclic: 509.599
mu coefficients   -- cyclic: {'(Intercept)': 0.2119267524316286, 'x1': 2.0097008919499895, 'x2': -0.5182416380905119, 'x3': -0.034129394722980336}
mu coefficients   -- noncyclic: {'(Intercept)': 0.20321801831335037, 'x1': 1.9161669874094818, 'x2': -0.4424465466186893, 'x3': -0.00500971293390438}


/tmp/ipykernel_67697/3359926779.py:17: FutureWarning: fit.selected(...) on a distributional model is deprecated since mboostrs 0.1.4 and will be removed in 0.3.0. Write `fit.selected` for the whole mapping, or `fit.selected_which(parameter=...)` to filter it. The three model classes disagreed about which of these were methods and which were properties, so the same expression worked on one fit and returned a bound method on another. They are all properties now: they read stored state and take no arguments.
  sel_mu = m_noncyclic.selected(parameter="mu")
/tmp/ipykernel_67697/3359926779.py:18: FutureWarning: fit.selected(...) on a distributional model is deprecated since mboostrs 0.1.4 and will be removed in 0.3.0. Write `fit.selected` for the whole mapping, or `fit.selected_which(parameter=...)` to filter it. The three model classes disagreed about which of these were methods and which were properties, so the same expression worked on one fit and returned a bound method on another. They 

Non-cyclic fitting has slightly higher final risk in this example. Both
schedules use 400 total updates: cyclic fitting assigns 200 to $\mu$ and
200 to $\sigma$, while non-cyclic fitting assigns them according to the
candidate reduction in joint risk. The different allocations give close but
unequal $\mu$ coefficients and different final fitted models.

### Case study: child stunting in India

`gamboostLSS`'s original worked example {cite}`hofner2014gamboostlss` models
child stunting (a standardized height-for-age z-score, lower = more
stunted) as a function of the child's age and BMI and the mother's age and
BMI, allowing **both** the mean and the variability of stunting to depend on
these covariates — the modelling motivation from
[](gamlss_framework.ipynb) §1.3, applied to a real public-health question.


In [4]:
from collections import Counter

india = mb.datasets.india()
m_india = mb.gamboostLSS(
    "stunting ~ bbs(cage, df=4) + bbs(mbmi, df=4) + bbs(mage, df=4)",
    data=india, families=mb.GaussianLSS(),
    control=mb.boost_control(mstop=200, nu=0.1),
)

mu_freq = Counter(m_india.selected(parameter="mu"))
sigma_freq = Counter(m_india.selected(parameter="sigma"))
print("mu (mean stunting) selection frequency:  ", dict(mu_freq))
print("sigma (stunting variability) selection frequency:", dict(sigma_freq))


mu (mean stunting) selection frequency:   {'cage': 110, 'mbmi': 51, 'mage': 39}
sigma (stunting variability) selection frequency: {'cage': 56, 'mage': 83, 'mbmi': 61}


/tmp/ipykernel_67697/2275936021.py:10: FutureWarning: fit.selected(...) on a distributional model is deprecated since mboostrs 0.1.4 and will be removed in 0.3.0. Write `fit.selected` for the whole mapping, or `fit.selected_which(parameter=...)` to filter it. The three model classes disagreed about which of these were methods and which were properties, so the same expression worked on one fit and returned a bound method on another. They are all properties now: they read stored state and take no arguments.
  mu_freq = Counter(m_india.selected(parameter="mu"))
/tmp/ipykernel_67697/2275936021.py:11: FutureWarning: fit.selected(...) on a distributional model is deprecated since mboostrs 0.1.4 and will be removed in 0.3.0. Write `fit.selected` for the whole mapping, or `fit.selected_which(parameter=...)` to filter it. The three model classes disagreed about which of these were methods and which were properties, so the same expression worked on one fit and returned a bound method on another.

Child age (`cage`) dominates the mean-stunting selection frequency — stunting
worsens with age in this cohort, the well-documented pattern that motivated
the original study — while all three covariates contribute to the
*variability* of stunting, matching the qualitative finding in
{cite}`hofner2014gamboostlss` that mean and dispersion are driven by
different combinations of predictors. A mean-only model would have missed
the dispersion half of the finding.


## Summary

- Distributional (GAMLSS) boosting fits $K$ additive predictors, one per
  distribution parameter, by cycling through parameters cyclically or
  non-cyclically. The loss is the joint negative log-likelihood.
- Each parameter family provides per-parameter offsets, partial gradients
  (scores), and an inverse link. Both the Rust unit tests
  (`assert_ngradient_matches_finite_diff`) and this chapter's own executed
  check confirm every implemented family's gradient against a
  finite-difference of its risk.
- Cyclic and non-cyclic fitting are different optimization paths (demonstrated
  above). Both schedules accept linear `Bols`, difference-penalized `Bbs`,
  ridge-penalized `Bns`, random-intercept `Brandom`, graph-smoothed `Bmrf`, tensor-product `Bspatial`, shape-constrained `Bmono`, caller-constrained `Bconstrained`, and tree-based `Btree` learners in a
  `gamboostLSS` parameter formula.
- Cross-validation must respect path dependence. Diagonal grid points, where
  every parameter has the same budget, can share checkpoints from one cyclic
  fit. A non-diagonal point is refitted because freezing one parameter earlier
  changes the later gradients of the others. Stability selection instead uses
  noncyclic subsample paths and records parameter-qualified learner identities.

## See also

- [](gamlss_framework.ipynb) — the non-boosting GAMLSS framework (the model class
  that boosting estimates).
- [](../implementation/lss_engine) — the LSS engine internals: sequential
  offsets, per-fold learner rebuild, and the `LssFamily` trait.
- [](../implementation/parity_and_bugs) — documented differences from R
  `gamboostLSS` (LogLogLSS σ offset, WeibullLSS μ offset, ZINBLSS dNBI
  recycling).
